In [12]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

In [13]:
df=pd.read_csv('/content/medical_data.csv')

In [14]:
df.head()

,Patient_Problem,Disease,Prescription
0,"Constant fatigue and muscle weakness, struggli...",Chronic Fatigue Syndrome,"Cognitive behavioral therapy, graded exercise ..."
1,"Frequent severe migraines, sensitivity to ligh...",Migraine with Aura,"Prescription triptans, avoid triggers like bri..."
2,"Sudden weight gain and feeling cold, especiall...",Hypothyroidism,Levothyroxine to regulate thyroid hormone levels.
3,"High fever, sore throat, and swollen lymph nod...",Mononucleosis,"Rest and hydration, ibuprofen for pain."
4,"Excessive thirst and frequent urination, dry m...",Diabetes Mellitus,Insulin therapy and lifestyle changes.


In [15]:
#Tokenization
tokenizer=Tokenizer(num_words=5000,oov_token="<OOv>")
tokenizer.fit_on_texts(df['Patient_Problem'])
sequences=tokenizer.texts_to_sequences(df['Patient_Problem'])

In [16]:
#Padding Sequences
max_len=max(len(seq) for seq in sequences)
padded_sequences=pad_sequences(sequences,maxlen=max_len,padding='post')

In [17]:
#Encoding
label_encoder_disease=LabelEncoder()
label_encoder_prescription=LabelEncoder()

disease_labels=label_encoder_disease.fit_transform(df['Disease'])
prescription_labels=label_encoder_prescription.fit_transform(df['Prescription'])

#Converting labels to categorical
disease_labels_categorical=to_categorical(disease_labels)
prescription_labels_categorical=to_categorical(prescription_labels)

In [18]:
Stack=np.hstack((disease_labels_categorical,prescription_labels_categorical))

In [19]:
# Model Building
input_layer=Input(shape=(max_len,))
embedding=Embedding(input_dim=5000,output_dim=64)(input_layer)
lstm_layer=LSTM(64)(embedding)
disease_output=Dense(len(label_encoder_disease.classes_),activation='softmax',name='disease_output')(lstm_layer)
prescription_output=Dense(len(label_encoder_prescription.classes_),activation='softmax',name='prescription_output')(lstm_layer)

In [20]:
model=Model(inputs=input_layer,outputs=[disease_output,prescription_output])
model.compile(
    loss={'disease_output':'categorical_crossentropy','prescription_output':'categorical_crossentropy'},
    optimizer='adam',
    metrics={'disease_output':['accuracy'],'prescription_output':['accuracy']}
)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 17)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 17, 64)    │    320,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     33,024 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ disease_output      │ (None, 178)       │     11,570 │ lstm_2[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prescription_output │ (None, 388)       │     25,220 │ lstm_2[0][0]      │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 389,814 (1.49 MB)

 Trainable params: 389,814 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
#Model Training
model.fit(padded_sequences,{'disease_output':disease_labels_categorical, 'prescription_output':prescription_labels_categorical},epochs=100,batch_size=32)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - disease_output_accuracy: 0.0295 - disease_output_loss: 5.1796 - loss: 11.1459 - prescription_output_accuracy: 0.0000e+00 - prescription_output_loss: 5.9665
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - disease_output_accuracy: 0.0344 - disease_output_loss: 5.1615 - loss: 11.1244 - prescription_output_accuracy: 0.0049 - prescription_output_loss: 5.9632
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - disease_output_accuracy: 0.0344 - disease_output_loss: 5.1096 - loss: 11.0766 - prescription_output_accuracy: 0.0074 - prescription_output_loss: 5.9683
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - disease_output_accuracy: 0.0344 - disease_output_loss: 4.9736 - loss: 10.9714 - prescription_output_accuracy: 0.0074 - prescription_output_loss: 6.0024
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - disease_output_accuracy: 0.0344 - disease_output_loss: 4.8795 - loss: 10.8407 - prescription_output_accuracy: 0.0123 -

In [23]:
def make_prediction(patient_problem):
  #Preprocessing the text
  sequence=tokenizer.texts_to_sequences([patient_problem])
  padded_sequences=pad_sequences(sequence,maxlen=max_len,padding='post')

  #Prediction
  prediction=model.predict(padded_sequences)

  # Prediction decoding
  disease_index=np.argmax(prediction[0],axis=1)[0]
  prescription_index=np.argmax(prediction[0],axis=1)[0]

  disease_predicted=label_encoder_disease.inverse_transform([disease_index])[0]
  prescription_predicted=label_encoder_prescription.inverse_transform([prescription_index])[0]

  print(f"Disease Predicted: {disease_predicted}")
  print(f"Prescription Predicted: {prescription_predicted}")

patient_input = "I've experienced a loss of appetite and don't enjoy food anymore."
make_prediction(patient_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
Disease Predicted: Depression
Prescription Predicted: Anticoagulants, avoid prolonged periods of immobility.
